# Phase 1: Best Algorithm Inspection & Gallery
Inspect and compare the top algorithms produced by each model.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
import difflib
import textwrap
from IPython.display import Markdown, display

plt.rcParams.update({"figure.figsize": (12, 6), "font.size": 11})

# --- Constants ---
RESULTS_DIR = Path("../results_phase1")
MODELS = [
    "qwen3.5-4b", "qwen3.5-9b", "qwen3.5-27b",
    "rnj-1-8b", "devstral-small-2-24b",
    "olmo3-7b", "olmo3-32b",
    "granite4-3b",
    "gemini-3-pro", "gemini-3-flash",
]
N_SEEDS = 5
BUDGET = 100
N_INSTANCES = 10
EVAL_SEEDS = 5


def parse_fitness(val):
    try:
        return float(val)
    except (TypeError, ValueError):
        return float("-inf")


def load_run(run_dir: Path):
    """Load all entries from a run's log.jsonl."""
    log_file = run_dir / "log.jsonl"
    if not log_file.exists():
        return []
    entries = []
    with open(log_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            entries.append(json.loads(line))
    return entries


# --- Load best algorithms ---
best_per_model = {}   # model -> dict with best candidate info
all_best = []         # list of dicts for statistical comparison

for model in MODELS:
    model_dir = RESULTS_DIR / model
    if not model_dir.exists():
        print(f"WARNING: {model_dir} not found, skipping.")
        continue

    seed_bests = []  # (seed, best_entry, best_fitness, aucs)
    for seed in range(N_SEEDS):
        seed_dir = model_dir / f"seed-{seed}"
        if not seed_dir.exists():
            continue
        run_dirs = sorted(seed_dir.glob("run-*"))
        if not run_dirs:
            continue
        # Use the last (or only) run directory
        entries = load_run(run_dirs[-1])
        if not entries:
            continue
        # Find best entry in this seed
        best_entry = None
        best_fit = float("-inf")
        for e in entries:
            f = parse_fitness(e.get("fitness"))
            if f > best_fit:
                best_fit = f
                best_entry = e
        if best_entry is not None:
            aucs = best_entry.get("metadata", {}).get("aucs", [])
            seed_bests.append((seed, best_entry, best_fit, aucs))
            all_best.append({
                "model": model,
                "seed": seed,
                "fitness": best_fit,
                "name": best_entry.get("name", "?"),
                "aucs": aucs,
            })

    if not seed_bests:
        continue

    # Pick the seed with the highest best AOCC
    seed_bests.sort(key=lambda x: x[2], reverse=True)
    best_seed, best_entry, best_fit, aucs = seed_bests[0]
    aucs_arr = np.array(aucs) if aucs else np.array([])
    best_per_model[model] = {
        "seed": best_seed,
        "fitness": best_fit,
        "name": best_entry.get("name", "?"),
        "code": best_entry.get("code", ""),
        "aucs": aucs_arr,
        "mean_aocc": float(np.mean(aucs_arr)) if len(aucs_arr) > 0 else float("-inf"),
        "std_aocc": float(np.std(aucs_arr)) if len(aucs_arr) > 0 else 0.0,
    }

# --- Summary table ---
rows = []
for model in MODELS:
    if model not in best_per_model:
        continue
    info = best_per_model[model]
    rows.append({
        "Model": model,
        "Algorithm": info["name"],
        "Best AOCC (mean +/- std)": f"{info['mean_aocc']:.4f} +/- {info['std_aocc']:.4f}",
        "Seed": info["seed"],
    })

df_summary = pd.DataFrame(rows)
display(df_summary)

# --- Display code for each model's best algorithm ---
for model in MODELS:
    if model not in best_per_model:
        continue
    info = best_per_model[model]
    md = f"### {model} — {info['name']} (AOCC={info['mean_aocc']:.4f}, seed={info['seed']})\n\n"
    md += f"```python\n{info['code']}\n```\n"
    display(Markdown(md))

In [ ]:
# --- Per-Instance Performance Heatmap ---
# Reshape 50 AUCs -> (10 instances x 5 eval seeds), take mean per instance

# Sort models by overall best AOCC
sorted_models = sorted(
    [m for m in MODELS if m in best_per_model],
    key=lambda m: best_per_model[m]["mean_aocc"],
    reverse=True,
)

instance_matrix = []
model_labels = []

for model in sorted_models:
    info = best_per_model[model]
    aucs = info["aucs"]
    if len(aucs) != N_INSTANCES * EVAL_SEEDS:
        # Pad or skip
        print(f"WARNING: {model} has {len(aucs)} AUCs, expected {N_INSTANCES * EVAL_SEEDS}")
        instance_means = np.full(N_INSTANCES, np.nan)
    else:
        # Reshape: (N_INSTANCES, EVAL_SEEDS)
        aucs_reshaped = aucs.reshape(N_INSTANCES, EVAL_SEEDS)
        instance_means = aucs_reshaped.mean(axis=1)
    instance_matrix.append(instance_means)
    model_labels.append(f"{model} ({info['mean_aocc']:.3f})")

instance_matrix = np.array(instance_matrix)

fig, ax = plt.subplots(figsize=(14, max(6, len(sorted_models) * 0.6)))
im = ax.imshow(instance_matrix, aspect="auto", cmap="RdYlGn")

# Text annotations
for i in range(instance_matrix.shape[0]):
    for j in range(instance_matrix.shape[1]):
        val = instance_matrix[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.3f}", ha="center", va="center", fontsize=9,
                    color="black" if 0.3 < val < 0.8 else "white")

ax.set_xticks(range(N_INSTANCES))
ax.set_xticklabels([f"Inst {i}" for i in range(N_INSTANCES)])
ax.set_yticks(range(len(model_labels)))
ax.set_yticklabels(model_labels)
ax.set_xlabel("MA-BBOB Instance")
ax.set_ylabel("Model (overall AOCC)")
ax.set_title("Per-Instance AOCC Heatmap (best algorithm per model)")
plt.colorbar(im, ax=ax, label="AOCC")
plt.tight_layout()
plt.savefig("phase1_instance_heatmap.pdf", bbox_inches="tight")
plt.show()

# Identify hard/easy instances
mean_per_instance = np.nanmean(instance_matrix, axis=0)
print("\nMean AOCC per instance across models:")
for i, m in enumerate(mean_per_instance):
    label = "(HARDEST)" if i == np.nanargmin(mean_per_instance) else \
            "(EASIEST)" if i == np.nanargmax(mean_per_instance) else ""
    print(f"  Instance {i}: {m:.4f} {label}")

In [ ]:
# --- Pairwise Algorithm Comparison ---

def cliffs_delta(x, y):
    n_x, n_y = len(x), len(y)
    more = sum(1 for xi in x for yi in y if xi > yi)
    less = sum(1 for xi in x for yi in y if xi < yi)
    return (more - less) / (n_x * n_y)


n_models = len(sorted_models)
pval_matrix = np.ones((n_models, n_models))
delta_matrix = np.zeros((n_models, n_models))
win_matrix = np.zeros((n_models, n_models), dtype=int)

for i in range(n_models):
    aucs_i = best_per_model[sorted_models[i]]["aucs"]
    for j in range(i + 1, n_models):
        aucs_j = best_per_model[sorted_models[j]]["aucs"]
        if len(aucs_i) == 0 or len(aucs_j) == 0:
            continue
        min_len = min(len(aucs_i), len(aucs_j))
        ai, aj = aucs_i[:min_len], aucs_j[:min_len]

        # Wilcoxon signed-rank test
        try:
            stat, p = stats.wilcoxon(ai, aj)
        except ValueError:
            p = 1.0
        pval_matrix[i, j] = p
        pval_matrix[j, i] = p

        # Cliff's delta
        d = cliffs_delta(ai, aj)
        delta_matrix[i, j] = d
        delta_matrix[j, i] = -d

        # Win/loss counts
        wins_i = int(np.sum(ai > aj))
        wins_j = int(np.sum(aj > ai))
        win_matrix[i, j] = wins_i
        win_matrix[j, i] = wins_j

# --- Combined heatmap: p-values (lower tri) + win counts (upper tri) ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

short_labels = sorted_models

# P-value heatmap
ax = axes[0]
mask = np.triu(np.ones_like(pval_matrix, dtype=bool), k=0)
pval_display = np.ma.masked_where(mask, pval_matrix)
im0 = ax.imshow(pval_display, cmap="RdYlGn_r", vmin=0, vmax=0.1)
for i in range(n_models):
    for j in range(i):
        val = pval_matrix[i, j]
        marker = "*" if val < 0.05 else ""
        ax.text(j, i, f"{val:.3f}{marker}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(n_models))
ax.set_xticklabels(short_labels, rotation=45, ha="right")
ax.set_yticks(range(n_models))
ax.set_yticklabels(short_labels)
ax.set_title("Wilcoxon p-values (lower triangle)")
plt.colorbar(im0, ax=ax, label="p-value", shrink=0.8)

# Cliff's delta heatmap
ax = axes[1]
im1 = ax.imshow(delta_matrix, cmap="RdBu", vmin=-1, vmax=1)
for i in range(n_models):
    for j in range(n_models):
        if i == j:
            continue
        val = delta_matrix[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(n_models))
ax.set_xticklabels(short_labels, rotation=45, ha="right")
ax.set_yticks(range(n_models))
ax.set_yticklabels(short_labels)
ax.set_title("Cliff's delta (row vs column, +1 = row dominates)")
plt.colorbar(im1, ax=ax, label="Cliff's delta", shrink=0.8)

plt.tight_layout()
plt.savefig("phase1_pairwise_comparison.pdf", bbox_inches="tight")
plt.show()

# Print win matrix
print("\nWin counts (row beat column on N of 50 paired AUCs):")
df_wins = pd.DataFrame(win_matrix, index=short_labels, columns=short_labels)
display(df_wins)

In [ ]:
# --- Code Similarity Analysis ---
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform

n_models = len(sorted_models)
sim_matrix = np.zeros((n_models, n_models))

# Compute upper triangle only (symmetric)
for i in range(n_models):
    code_i = best_per_model[sorted_models[i]]["code"]
    sim_matrix[i, i] = 1.0
    for j in range(i + 1, n_models):
        code_j = best_per_model[sorted_models[j]]["code"]
        ratio = difflib.SequenceMatcher(None, code_i, code_j).ratio()
        sim_matrix[i, j] = ratio
        sim_matrix[j, i] = ratio

# --- Similarity heatmap ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
im = ax.imshow(sim_matrix, cmap="YlOrRd", vmin=0, vmax=1)
for i in range(n_models):
    for j in range(n_models):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(n_models))
ax.set_xticklabels(sorted_models, rotation=45, ha="right")
ax.set_yticks(range(n_models))
ax.set_yticklabels(sorted_models)
ax.set_title("Code Similarity (SequenceMatcher ratio)")
plt.colorbar(im, ax=ax, label="Similarity", shrink=0.8)

# --- Dendrogram ---
ax = axes[1]
dist_matrix = 1.0 - sim_matrix
np.fill_diagonal(dist_matrix, 0)  # ensure diagonal is exactly 0
condensed_dist = squareform(dist_matrix)
Z = linkage(condensed_dist, method="average")
dendrogram(Z, labels=sorted_models, ax=ax, leaf_rotation=45, leaf_font_size=10)
ax.set_title("Code Similarity Dendrogram (average linkage)")
ax.set_ylabel("Distance (1 - similarity)")

plt.tight_layout()
plt.savefig("phase1_code_similarity.pdf", bbox_inches="tight")
plt.show()

# --- Diff statistics ---
print("\nPairwise diff statistics (unified_diff changed lines / total lines):")
for i in range(n_models):
    code_i = best_per_model[sorted_models[i]]["code"].splitlines()
    for j in range(i + 1, n_models):
        code_j = best_per_model[sorted_models[j]]["code"].splitlines()
        diff = list(difflib.unified_diff(code_i, code_j, lineterm=""))
        changed = sum(1 for line in diff if line.startswith("+") or line.startswith("-"))
        total = max(len(code_i), len(code_j), 1)
        ratio = changed / total
        print(f"  {sorted_models[i]:>25s} vs {sorted_models[j]:<25s}: "
              f"sim={sim_matrix[i,j]:.3f}  diff_ratio={ratio:.3f}  "
              f"changed_lines={changed}")

# Summary
upper_sims = sim_matrix[np.triu_indices(n_models, k=1)]
print(f"\nOverall code similarity stats (N={len(upper_sims)} pairs):")
print(f"  Mean: {np.mean(upper_sims):.3f}")
print(f"  Std:  {np.std(upper_sims):.3f}")
print(f"  Min:  {np.min(upper_sims):.3f}")
print(f"  Max:  {np.max(upper_sims):.3f}")